In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import xarray as xr

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/thp/')

In [ ]:
%reload_ext autoreload
%autoreload 2

## Directories and variables

In [ ]:
workdir = '/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/thp'
zarr_dir = f'{workdir}/zarr_stores'
figdir = '/uufs/chpc.utah.edu/common/home/u6058223/public_html/thp_update/melt_figures'
basins    = ['animas', 'yampa', 'jordan']
wys       = [2022, 2023, 2024]
eb_terms  = ['net_solar', 'net_LW', 'sensible_heat', 'latent_heat',
                  'snow_soil', 'precip_advected']  # adjust to your suffix
suffix    = 'split'
elev_bins = np.arange(1200, 4800, 200)

## Bar chart of total seasonal SWI by EB term

In [ ]:
# h.fn_list(f'{zarr_dir}', f'{basin}_eb_proportions_{suffix}_swi_attributed.nc)
proportions_fns = h.fn_list(f'{zarr_dir}', '*_eb_proportions_*_swi_attributed.nc')

ds = xr.open_dataset(proportions_fns[0])
ds['swi_attributed_taf'].plot()

In [ ]:
from melt_plots import plot_swi_eb_total, _abbrev

# --- Load attributed TAF values ---
# swi_attributed_taf has dims (term, elev_bin); sum over elev_bin for basin total
attr_data = {}
for basin in basins:
    for wy in wys:
        fn = f'{zarr_dir}/{basin}_eb_proportions_{suffix}_wy{wy}_swi_attributed.nc'
        ds = xr.open_dataset(fn)
        # Build swi_by_term dict with (N_ASPECT_BINS,)-like structure — here just scalar per term
        attr_data[(basin, wy)] = {
            t: ds['swi_attributed_taf'].sel(term=t).values  # (elev_bin,)
            for t in eb_terms
        }
# --- Per-basin xlim: max total SWI across all WYs ---
xlim_per_basin = {}
for basin in basins:
    max_total = max(
        sum(attr_data[(basin, wy)][t].sum() for t in eb_terms)*1.02
        for wy in wys
    )
    xlim_per_basin[basin] = (0, max_total * 1.15)

# --- Megafigure ---
fig, axes = plt.subplots(
    len(wys), len(basins),
    figsize=(15, 6),
    sharex='col',    # share x per column so per-basin scale is consistent
)

for i, wy in enumerate(wys):
    for j, basin in enumerate(basins):
        plot_swi_eb_total(
            attr_data[(basin, wy)], eb_terms, wy, basin,
            xlim=xlim_per_basin[basin],
            specify_ax=(fig, axes[i, j]),
            tick_fontsize=11,
            precomputed=True,
            annotate=True,
            label_pct_threshold=0.0
        )
        if j > 0:
            axes[i, j].set_ylabel('')
        if i < len(wys) - 1:
            axes[i, j].set_xlabel('')

# Shared legend at bottom
colors = plt.get_cmap('Set2')(np.linspace(0, 1, len(eb_terms)))
handles = [mpatches.Patch(color=colors[k], label=_abbrev(t), linewidth=0)
           for k, t in enumerate(eb_terms)]
fig.legend(handles=handles, loc='lower center',
           bbox_to_anchor=(0.5, -0.04), ncol=len(eb_terms),
           fontsize=14, handlelength=0.8, frameon=False)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(f'{figdir}/megafig_swi_eb_total.png', dpi=300, bbox_inches='tight')


## EB median contributions heatmap [DOWY × elevation]

In [ ]:
# h.fn_list(f'{zarr_dir}', f'{basin}_eb_proportions_{suffix}_swi_attributed.nc)
contributions_fns = h.fn_list(f'{zarr_dir}', '*_eb_proportions_*_medians.nc')
# contributions_fns
ds = xr.open_dataset(contributions_fns[0])
ds['median_fraction'].sel(term='net_solar').T.plot(figsize=(5,2), cmap='Purples')

In [ ]:
ds.term.values

In [ ]:
del ds

In [ ]:
term = 'snow_soil'
# --- Load all cached median matrices ---
medians = {}
for basin in basins:
    for wy in wys:
        fn = f'{zarr_dir}/{basin}_eb_proportions_{suffix}_wy{wy}_medians.nc'
        ds = xr.open_dataset(fn)
        medians[(basin, wy)] = ds['median_fraction'].sel(term=term).values  # (dowy, elev_bin)
        dowy = ds['dowy'].values   # same across all files

In [ ]:
from melt_plots import plot_eb_term_heatmap, Normalize
# --- Build megafigure: rows=WY, cols=basin ---
fig, axes = plt.subplots(
    len(wys), len(basins),
    # figsize=(5 * len(basins), 2 * len(wys)),
    figsize=(15,6),
    sharey=True, sharex=True,
)

for i, wy in enumerate(wys):
    for j, basin in enumerate(basins):
        ax = axes[i, j]
        plot_eb_term_heatmap(
            medians[(basin, wy)], term, dowy, elev_bins, wy, basin,
            specify_ax=(fig, ax), annotate=False, 
            tick_fontsize=12,
        )
        # Only label axes on the edges
        if j > 0:
            ax.set_ylabel('')
        if i < len(wys) - 1:
            ax.set_xticklabels([])

# Shared colorbar on top
sm = plt.cm.ScalarMappable(cmap='Purples', norm=Normalize(0, 1))
sm.set_array([])
# Move the colorbar outside of the figure at the top
cbar_ax = fig.add_axes([0.25, 1.2, 0.5, 0.3]) # [left, bottom, width, height]
cb = fig.colorbar(sm, ax=cbar_ax, orientation='horizontal',
             label='Median fractional contribution')
cb.ax.tick_params(labelsize=16)
# Increase the font size of the colorbar label
cb.ax.xaxis.label.set_size(16)
# Turn off the axes for the colorbar
cbar_ax.set_axis_off()
# fig.suptitle(f'Median EB contribution — {term}', y=1.01)
plt.tight_layout();
plt.savefig(f'{figdir}/megafig_{term}.png', dpi=300, bbox_inches='tight')

In [ ]:
# Make some space in memory
del medians

## Cumulative seasonal SWI (TAF) heatmap [DOWY x elevation]

In [ ]:
from melt_plots import plot_variable_elevation_heatmap

# Load all cached swi_sum_taf NetCDFs
swi_data = {}
pixels_data = {}
for basin in basins:
    for wy in wys:
        ds = xr.open_dataset(f'{zarr_dir}/{basin}_swi_sum_taf_wy{wy}.nc')
        swi_data[(basin, wy)] = ds['swi_taf'].values  # (dowy, elev_bin)
        pixels_data[(basin, wy)] = ds['pixels_per_bin'].values  # (elev_bin,)
        dowy     = ds['dowy'].values
        elev_bin = ds['elev_bin'].values

In [ ]:
vmax_per_basin = {'animas': 5, 'yampa': 11, 'jordan': 3}

fig, axes = plt.subplots(len(wys), len(basins),
                          figsize=(15,6*1.1),
                          sharey=True, sharex=True)

for i, wy in enumerate(wys):
    for j, basin in enumerate(basins):
        ax = axes[i, j]
        plot_variable_elevation_heatmap(
            swi_data[(basin, wy)], 'SWI', dowy, elev_bins, wy, basin,
            units='precomputed', aggregation='sum',
            vmin=0, vmax=vmax_per_basin[basin],
            pixels_per_bin=pixels_data[(basin, wy)],
            specify_ax=(fig, ax), annotate=False,
            tick_fontsize=12,
        )
        if j > 0:
            ax.set_ylabel('')
        if i < len(wys) - 1:
            ax.set_xticklabels([])

# Leave room at the top for the colorbars
plt.tight_layout(rect=[0, 0, 1, 0.91])

# One colorbar per basin column — positioned using actual subplot bounds
for j, basin in enumerate(basins):
    pos = axes[0, j].get_position()         # bounding box of top subplot in column
    # [left, bottom, width, height] in figure-normalised coordinates
    cbar_ax = fig.add_axes([pos.x0 + pos.width*0.75/10, pos.y1 + 0.1, pos.width*0.75, 0.025])

    sm = plt.cm.ScalarMappable(
        cmap='YlGnBu',
        norm=Normalize(0, vmax_per_basin[basin])
    )
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
    cb.ax.tick_params(labelsize=12)
    cb.set_label(f'{basin.capitalize()} SWI (TAF)', fontsize=12)

In [ ]:
del swi_data, pixels_data